# 예제 05. 네 모델 실험과 결과 분석
빅데이터프로그래밍 · 9주차

## 목표
- 기본 · Dropout · BatchNorm · 데이터 증강 CNN을 같은 조건에서 비교한다
- 정확도뿐 아니라 학습 시간과 곡선, 오류 이미지까지 확인한다
- 결과표를 만들어 어느 모델을 고를지 판단한다

과제와 같은 형식입니다. 이 노트북을 그대로 확장하면 과제가 됩니다.

**런타임 > 런타임 유형 변경 > T4 GPU** 를 먼저 선택하세요.


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import pandas as pd
import time

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


## 1. 실험 조건을 고정합니다
비교하려면 **한 가지만** 바꿔야 합니다. 나머지는 모두 같게 둡니다.


In [ ]:
EPOCHS = 25
BATCH = 64
LR = 1e-3
N_TRAIN = 2000
SEED = 42

test_tf = transforms.ToTensor()
aug_tf = transforms.Compose([
    transforms.RandomRotation(15),
    transforms.RandomHorizontalFlip(),
    transforms.RandomAffine(0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ToTensor(),
])

plain_train = datasets.FashionMNIST("./data", train=True,  download=True, transform=test_tf)
aug_train   = datasets.FashionMNIST("./data", train=True,  download=True, transform=aug_tf)
test_set    = datasets.FashionMNIST("./data", train=False, download=True, transform=test_tf)

plain_loader = DataLoader(Subset(plain_train, range(N_TRAIN)), batch_size=BATCH, shuffle=True)
aug_loader   = DataLoader(Subset(aug_train,   range(N_TRAIN)), batch_size=BATCH, shuffle=True)
test_loader  = DataLoader(test_set, batch_size=256, shuffle=False)

CLASSES = ["티셔츠", "바지", "풀오버", "드레스", "코트",
           "샌들", "셔츠", "운동화", "가방", "앵클부츠"]
print(f"epoch {EPOCHS} · batch {BATCH} · lr {LR} · 학습 {N_TRAIN}장")


## 2. 네 가지 모델


In [ ]:
def make_cnn(dropout=0.0, batchnorm=False, n_classes=10):
    def conv_block(cin, cout):
        layers = [nn.Conv2d(cin, cout, 3, padding=1)]
        if batchnorm:
            layers.append(nn.BatchNorm2d(cout))
        layers += [nn.ReLU(), nn.MaxPool2d(2)]
        return layers

    head = [nn.Flatten()]
    if dropout:
        head.append(nn.Dropout(dropout))
    head += [nn.Linear(64*7*7, 256)]
    if batchnorm:
        head.append(nn.BatchNorm1d(256))
    head.append(nn.ReLU())
    if dropout:
        head.append(nn.Dropout(dropout))
    head.append(nn.Linear(256, n_classes))

    return nn.Sequential(*conv_block(1, 32), *conv_block(32, 64), *head)


configs = [
    ("기본 CNN",       dict(dropout=0.0, batchnorm=False), plain_loader),
    ("Dropout CNN",    dict(dropout=0.5, batchnorm=False), plain_loader),
    ("BatchNorm CNN",  dict(dropout=0.0, batchnorm=True),  plain_loader),
    ("데이터 증강 CNN", dict(dropout=0.0, batchnorm=False), aug_loader),
]

for name, kw, _ in configs:
    m = make_cnn(**kw)
    print(f"{name:16s} 파라미터 {sum(p.numel() for p in m.parameters()):,}개")


## 3. 학습


In [ ]:
loss_fn = nn.CrossEntropyLoss()

def measure(model, loader):
    model.eval()
    loss_sum = correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss_sum += loss_fn(out, y).item() * y.numel()
            correct += (out.argmax(dim=1) == y).sum().item()
            total += y.numel()
    return loss_sum / total, correct / total


results = {}
for name, kw, loader in configs:
    torch.manual_seed(SEED)
    model = make_cnn(**kw).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)

    print(f"\n{name}")
    start = time.time()
    hist = []
    for epoch in range(1, EPOCHS + 1):
        model.train()
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            loss = loss_fn(model(x), y)
            opt.zero_grad(); loss.backward(); opt.step()
        hist.append((*measure(model, plain_loader), *measure(model, test_loader)))
        if epoch % 10 == 0 or epoch == 1:
            print(f"  epoch {epoch:2d}  학습 {hist[-1][1]:.4f}  검증 {hist[-1][3]:.4f}")

    results[name] = {"model": model, "hist": hist, "time": time.time() - start}
    print(f"  학습 시간 {results[name]['time']:.1f}초")


## 4. 결과 비교표


In [ ]:
rows = []
for name in results:
    h = results[name]["hist"]
    rows.append({
        "모델": name,
        "학습 정확도": round(h[-1][1], 4),
        "검증 정확도": round(h[-1][3], 4),
        "차이": round(h[-1][1] - h[-1][3], 4),
        "최고 검증": round(max(x[3] for x in h), 4),
        "검증 손실": round(h[-1][2], 4),
        "학습 시간(초)": round(results[name]["time"], 1),
    })
table = pd.DataFrame(rows)
print(table.to_string(index=False))

best = table.loc[table["검증 정확도"].idxmax(), "모델"]
print(f"\n검증 정확도가 가장 높은 모델: {best}")


## 5. 학습 곡선 네 개 겹쳐 보기


In [ ]:
xs = range(1, EPOCHS + 1)
fig, ax = plt.subplots(2, 2, figsize=(13, 8))

for name in results:
    h = results[name]["hist"]
    ax[0,0].plot(xs, [v[0] for v in h], label=name)
    ax[0,1].plot(xs, [v[2] for v in h], label=name)
    ax[1,0].plot(xs, [v[3] for v in h], label=name)
    ax[1,1].plot(xs, [v[1]-v[3] for v in h], label=name)

titles = ["학습 손실", "검증 손실", "검증 정확도", "학습 − 검증 차이"]
for a, t in zip(ax.flatten(), titles):
    a.set_title(t); a.set_xlabel("epoch"); a.legend(fontsize=9); a.grid(alpha=.3)
plt.tight_layout(); plt.show()


## 6. 가장 좋은 모델의 오류 이미지


In [ ]:
model = results[best]["model"]
model.eval()

imgs, trues, preds = [], [], []
with torch.no_grad():
    for x, y in test_loader:
        p = model(x.to(device)).argmax(dim=1).cpu()
        wrong = p != y
        if wrong.any():
            imgs.append(x[wrong]); trues.append(y[wrong]); preds.append(p[wrong])
        if sum(len(t) for t in trues) > 16:
            break

imgs = torch.cat(imgs); trues = torch.cat(trues); preds = torch.cat(preds)

fig, axes = plt.subplots(2, 8, figsize=(16, 4.6))
for ax_, i in zip(axes.flatten(), range(16)):
    ax_.imshow(imgs[i].squeeze(), cmap="gray")
    ax_.set_title(f"{CLASSES[preds[i]]}\n(정답 {CLASSES[trues[i]]})", fontsize=9, color="crimson")
    ax_.axis("off")
plt.suptitle(f"{best} — 틀린 사례", y=1.02)
plt.tight_layout(); plt.show()


## 7. 클래스별 정확도 비교


In [ ]:
def per_class(model):
    model.eval()
    correct = torch.zeros(10); total = torch.zeros(10)
    with torch.no_grad():
        for x, y in test_loader:
            p = model(x.to(device)).argmax(dim=1).cpu()
            for c in range(10):
                mask = y == c
                total[c] += mask.sum(); correct[c] += (p[mask] == c).sum()
    return (correct / total).tolist()


cls_df = pd.DataFrame({"클래스": CLASSES})
for name in results:
    cls_df[name] = [round(v, 3) for v in per_class(results[name]["model"])]
print(cls_df.to_string(index=False))


## 8. 두 가지를 함께 쓰면
Dropout과 BatchNorm은 같이 쓸 수 있습니다. 증강까지 더하면 세 가지가 됩니다.


In [ ]:
torch.manual_seed(SEED)
combo = make_cnn(dropout=0.3, batchnorm=True).to(device)
opt = torch.optim.Adam(combo.parameters(), lr=LR)

start = time.time()
combo_hist = []
for epoch in range(EPOCHS):
    combo.train()
    for x, y in aug_loader:          # 증강까지
        x, y = x.to(device), y.to(device)
        loss = loss_fn(combo(x), y)
        opt.zero_grad(); loss.backward(); opt.step()
    combo_hist.append((*measure(combo, plain_loader), *measure(combo, test_loader)))

print("세 가지 모두  검증 정확도:", round(combo_hist[-1][3], 4),
      "· 차이", round(combo_hist[-1][1]-combo_hist[-1][3], 4),
      f"· {time.time()-start:.1f}초")
print(f"{best}      검증 정확도:", round(max(x[3] for x in results[best]['hist']), 4))


## 직접 해보기
1. 학습 데이터를 10,000장으로 늘리면 네 모델의 순위가 바뀌나요?
2. epoch을 50으로 늘리면 어떤 모델이 가장 잘 버티나요?
3. 학습 시간과 정확도를 함께 보면 어느 모델을 고르시겠습니까?


In [ ]:
# 여기에 작성하세요
